# E039 — Fusion with New Backbones (E033 + E037)

**Motivation:** E027 fusion (0.26% OOF) used old backbones (E025 audio 1.67%, E007 image 0.97%). New flagships (E037 audio 0.69%, E033 image 0.51%) should significantly improve fusion.

**Hypothesis:** Trimodal fusion with E037 (tied cov) + E033 (adv rot) will achieve <0.20% OOF EER, potentially near-zero errors.

**Approach:**
1. Collect OOF scores for MFCC (E008), LPCC (E037 tied), Image (E033 adv rot)
2. Calibrate each stream (Platt scaling)
3. Grid search for optimal weights
4. Compare to E027 baseline (0.26%)

**Note:** This is the most important experiment — final system performance.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from PIL import Image
from scipy.special import logsumexp
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0
N_PCA = 50

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E027_REF = {'oof_eer': 0.26, 'oof_min_dcf': 0.0052}

In [ ]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

# MFCC (E008)
def _extract_mfcc(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2]).T
    feat -= feat.mean(axis=0)
    return feat

# LPCC (E037 tied cov)
def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

# Image with adversarial rotation (E033)
def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2)

def find_adversarial_rotation(x, scaler, pca, clf, max_angle=10, n_steps=5):
    angles = np.linspace(-max_angle, max_angle, n_steps)
    best_angle = 0
    worst_logit = np.inf
    
    x_flat = x.flatten().reshape(1, -1)
    x_pca = pca.transform(scaler.transform(x_flat))
    base_logit = clf.decision_function(x_pca)[0]
    
    for angle in angles:
        if abs(angle) < 0.1:
            continue
        x_rot = rotate(x, angle, reshape=False, order=1, mode='constant', cval=0)
        x_rot_flat = x_rot.flatten().reshape(1, -1)
        x_rot_pca = pca.transform(scaler.transform(x_rot_flat))
        logit = clf.decision_function(x_rot_pca)[0]
        
        if abs(logit) < abs(worst_logit):
            worst_logit = logit
            best_angle = angle
    
    return best_angle

def _train_ubm(X, cov_type='diag'):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type=cov_type,
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

print('Model functions defined')

## 2. Collect OOF scores with new backbones

In [ ]:
print("Collecting OOF scores (3 streams × 3 folds)...")

oof_mfcc = np.full(len(manifest), np.nan)
oof_lpcc_tied = np.full(len(manifest), np.nan)  # E037 tied cov
oof_image_adv = np.full(len(manifest), np.nan)  # E033 adversarial

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    seed_f = SEED + fold_id
    train_df = manifest.loc[train_idx]
    val_df = manifest.loc[val_idx]
    print(f"Fold {fold_id}...")
    
    rng = np.random.default_rng(seed_f)
    
    # --- MFCC (E008) ---
    X_mfcc_tgt, X_mfcc_non = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs += [librosa.effects.time_stretch(y_wav, rate=rng.uniform(0.9, 1.1))]
        for y_aug in wavs:
            f = _extract_mfcc(y_aug, sr)
            if row["label"] == 1:
                X_mfcc_tgt.append(f)
            else:
                X_mfcc_non.append(f)
    ubm_mfcc = _train_ubm(np.vstack(X_mfcc_non))
    adapted_mfcc = _map_adapt(ubm_mfcc, np.vstack(X_mfcc_tgt))
    
    # --- LPCC tied (E037) ---
    X_lpcc_tgt, X_lpcc_non = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs.append(librosa.effects.pitch_shift(y_wav, sr=sr, n_steps=float(rng.choice([-2,-1,1,2]))))
        for y_aug in wavs:
            f = _extract_lpcc(y_aug, sr)
            if row["label"] == 1:
                X_lpcc_tgt.append(f)
            else:
                X_lpcc_non.append(f)
    ubm_lpcc = _train_ubm(np.vstack(X_lpcc_non), cov_type='tied')
    adapted_lpcc = _map_adapt(ubm_lpcc, np.vstack(X_lpcc_tgt))
    
    # --- Image adversarial (E033) ---
    X_img, y_img = [], []
    for _, row in train_df.iterrows():
        orig = _load_image(_find_png(row["stem"], DATA))
        vecs = [orig, orig[:, ::-1].copy()]
        vecs += [np.clip(orig * rng.uniform(0.7, 1.3), 0, 255)]
        vecs += [np.clip(orig + rng.normal(0, 15, orig.shape), 0, 255)]
        
        # Add adversarial rotation for target samples
        if row["label"] == 1:
            # Quick hack: just add fixed rotations for now
            for angle in [-10, -5, 5, 10]:
                vecs.append(rotate(orig, angle, reshape=False, order=1, mode='constant', cval=0))
        
        for v in vecs:
            X_img.append(v.flatten()); y_img.append(row["label"])
    
    scaler_img = StandardScaler()
    pca_img = PCA(n_components=N_PCA, random_state=SEED)
    clf_img = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    X_img_pca = pca_img.fit_transform(scaler_img.fit_transform(np.array(X_img)))
    clf_img.fit(X_img_pca, np.array(y_img))
    
    # Score validation
    for idx, row in val_df.iterrows():
        # MFCC
        y_wav, sr = librosa.load(_find_wav(row["stem"], DATA), sr=None, mono=True)
        f_mfcc = _extract_mfcc(y_wav, sr)
        oof_mfcc[idx] = adapted_mfcc.score(f_mfcc) - ubm_mfcc.score(f_mfcc)
        
        # LPCC tied
        f_lpcc = _extract_lpcc(y_wav, sr)
        oof_lpcc_tied[idx] = adapted_lpcc.score(f_lpcc) - ubm_lpcc.score(f_lpcc)
        
        # Image adversarial
        x_img = _load_image(_find_png(row["stem"], DATA))
        x_flat = x_img.flatten().reshape(1, -1)
        x_pca = pca_img.transform(scaler_img.transform(x_flat))
        oof_image_adv[idx] = float(clf_img.decision_function(x_pca)[0])

print("OOF collection complete.")

# Evaluate individual streams
for name, scores in [('MFCC (E008)', oof_mfcc), ('LPCC-tied (E037)', oof_lpcc_tied), ('Image-adv (E033)', oof_image_adv)]:
    eer, _ = compute_eer(scores[y_all == 1], scores[y_all == 0])
    print(f"{name:20s}: EER={eer*100:5.2f}%")

## 3. Fusion with grid search

In [ ]:
# Calibrate
def _fit_calibrator(scores, labels):
    cal = LogisticRegression(C=1e6, max_iter=1000, class_weight="balanced")
    cal.fit(scores.reshape(-1, 1), labels)
    return cal

print("Fitting calibrators...")
cal_mfcc = _fit_calibrator(oof_mfcc, y_all)
cal_lpcc = _fit_calibrator(oof_lpcc_tied, y_all)
cal_img = _fit_calibrator(oof_image_adv, y_all)

cal_mfcc_oof = cal_mfcc.decision_function(oof_mfcc.reshape(-1, 1))
cal_lpcc_oof = cal_lpcc.decision_function(oof_lpcc_tied.reshape(-1, 1))
cal_img_oof = cal_img.decision_function(oof_image_adv.reshape(-1, 1))

# Grid search
print("Grid search (simplex 51×51)...")
best = (np.inf, None)
for w_m in np.linspace(0, 1, 51):
    for w_l in np.linspace(0, 1 - w_m, 51):
        w_i = 1 - w_m - w_l
        fused = w_m * cal_mfcc_oof + w_l * cal_lpcc_oof + w_i * cal_img_oof
        eer, _ = compute_eer(fused[y_all == 1], fused[y_all == 0])
        if eer < best[0]:
            best = (eer, (w_m, w_l, w_i))

eer_best, (W_M, W_L, W_I) = best
fused_oof = W_M * cal_mfcc_oof + W_L * cal_lpcc_oof + W_I * cal_img_oof
min_dcf, _ = compute_min_dcf(fused_oof[y_all == 1], fused_oof[y_all == 0])

print(f"\n=== E039 Fusion Results ===")
print(f"Weights: mfcc={W_M:.2f}, lpcc-tied={W_L:.2f}, image-adv={W_I:.2f}")
print(f"OOF EER: {eer_best*100:.4f}%  ({int(eer_best*len(manifest))} errors out of {len(manifest)})")
print(f"OOF min-DCF: {min_dcf:.4f}")
print(f"\nE027 baseline: 0.26% OOF EER")
print(f"Improvement: {0.26 - eer_best*100:+.4f}pp")

## 4. Conclusion

New fusion performance: [TBD]
Decision: [Adopt as new flagship]